# Mina Dog Vocalization Classifier — Training

2-class model: **mina** (dog vocalizations) vs **negative** (ambient noise, speech, etc.)

Exports to TFLite for Raspberry Pi 5 inference (<25ms).

## Setup
1. Upload `training_data.zip` to your Google Drive root
2. Run cells in order — if Drive mount fails, run the "Fix Drive" cell and retry
3. Download `mina_classifier.tflite` when done

In [ ]:
# Install dependencies
!pip install -q librosa soundfile scikit-learn

In [ ]:
# Mount Google Drive and extract training data
from google.colab import drive
import zipfile, os

drive.mount('/content/drive')

# Look for training_data.zip in Drive root or a minazap folder
zip_paths = [
    '/content/drive/MyDrive/training_data.zip',
    '/content/drive/MyDrive/minazap/training_data.zip',
    '/content/drive/MyDrive/minazapper/training_data.zip',
]

zip_path = None
for p in zip_paths:
    if os.path.exists(p):
        zip_path = p
        break

if zip_path:
    print(f'Found: {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print('Extracted.')
else:
    print('training_data.zip not found in Drive. Searched:')
    for p in zip_paths:
        print(f'  {p}')
    print('Upload it to your Drive root and re-run this cell.')

# Verify
for d in ['training_data/mina', 'training_data/negative']:
    count = len([f for f in os.listdir(d) if f.endswith('.wav')]) if os.path.isdir(d) else 0
    print(f'{d}: {count} files')

### Fix Drive mount (only run if the cell above failed)

In [ ]:
# Run this if Drive mount failed with credential error, then re-run the cell above
from google.colab import runtime
runtime.unassign()  # Disconnects runtime — reconnect and re-run the mount cell

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Config
SAMPLE_RATE = 16000
WINDOW_SIZE = 16000
FRAME_LENGTH = 255
FRAME_STEP = 128
N_MELS = 40
N_MFCC = 40
LABELS = ['mina', 'negative']

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
def compute_mfcc(audio):
    """Compute MFCC features. Output shape: (124, 40)"""
    audio = audio.astype(np.float32)
    if len(audio) < WINDOW_SIZE:
        audio = np.pad(audio, (0, WINDOW_SIZE - len(audio)))
    elif len(audio) > WINDOW_SIZE:
        audio = audio[:WINDOW_SIZE]
    mfcc = librosa.feature.mfcc(
        y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
        n_fft=FRAME_LENGTH, hop_length=FRAME_STEP,
        n_mels=N_MELS, center=False,
    )
    return mfcc.T.astype(np.float32)  # (124, 40)

# Verify shape
test = compute_mfcc(np.random.randn(16000).astype(np.float32))
print(f'MFCC shape: {test.shape}')
assert test.shape == (124, 40)

In [ ]:
def load_audio(filepath):
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    if len(audio) < WINDOW_SIZE:
        audio = np.pad(audio, (0, WINDOW_SIZE - len(audio)))
    else:
        audio = audio[:WINDOW_SIZE]
    return audio.astype(np.float32)

def augment_time_shift(audio):
    shift = int(len(audio) * random.uniform(-0.1, 0.1))
    return np.roll(audio, shift)

def augment_gaussian_noise(audio, sigma=0.005):
    return audio + np.random.normal(0, sigma, len(audio)).astype(np.float32)

def augment_pitch_shift(audio):
    n_steps = random.uniform(-2, 2)
    return librosa.effects.pitch_shift(audio, sr=SAMPLE_RATE, n_steps=n_steps).astype(np.float32)

In [ ]:
# Load all audio
TRAINING_DATA_DIR = Path('./training_data')
all_audio = []
all_labels = []

for label_idx, label_name in enumerate(LABELS):
    label_dir = TRAINING_DATA_DIR / label_name
    if not label_dir.exists():
        print(f'WARNING: {label_dir} not found!')
        continue
    wav_files = sorted(label_dir.glob('*.wav'))
    print(f'Loading {label_name}: {len(wav_files)} files...')
    for wav_path in wav_files:
        try:
            audio = load_audio(str(wav_path))
            all_audio.append(audio)
            all_labels.append(label_idx)
        except Exception as e:
            pass

all_audio = np.array(all_audio, dtype=np.float32)
all_labels = np.array(all_labels, dtype=np.int32)
print(f'\nTotal samples: {len(all_labels)}')
for i, name in enumerate(LABELS):
    print(f'  {name}: {np.sum(all_labels == i)}')

In [ ]:
# Train/val/test split: 70/15/15
X_train_audio, X_temp_audio, y_train, y_temp = train_test_split(
    all_audio, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels
)
X_val_audio, X_test_audio, y_val, y_test = train_test_split(
    X_temp_audio, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print(f'Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}')

# Augment training data
print('Augmenting training data...')
X_train_aug = list(X_train_audio)
y_train_aug = list(y_train)

for i in range(len(X_train_audio)):
    audio = X_train_audio[i]
    for aug_fn in [augment_time_shift, augment_gaussian_noise, augment_pitch_shift]:
        X_train_aug.append(aug_fn(audio))
        y_train_aug.append(y_train[i])

X_train_audio_aug = np.array(X_train_aug, dtype=np.float32)
y_train_aug = np.array(y_train_aug, dtype=np.int32)
print(f'After augmentation: {len(y_train_aug)} training samples')

In [ ]:
# Compute MFCCs
print('Computing MFCCs (this may take a few minutes)...')
X_train = np.array([compute_mfcc(a) for a in X_train_audio_aug])[..., np.newaxis]
X_val = np.array([compute_mfcc(a) for a in X_val_audio])[..., np.newaxis]
X_test = np.array([compute_mfcc(a) for a in X_test_audio])[..., np.newaxis]

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')

In [ ]:
# Class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_aug), y=y_train_aug)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')

# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(124, 40, 1)),
    tf.keras.layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(LABELS), activation='softmax'),
])

model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
# Train
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    X_train, y_train_aug,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss: {test_loss:.4f}')

# Predictions
y_pred = np.argmax(model.predict(X_test), axis=1)

# Confusion matrix
print('\n--- Confusion Matrix ---')
cm = confusion_matrix(y_test, y_pred)
header = '          ' + '  '.join(f'{l:>8s}' for l in LABELS)
print(header)
for i, row in enumerate(cm):
    print(f'{LABELS[i]:>8s}  ' + '  '.join(f'{v:>8d}' for v in row))

print('\n--- Per-class Metrics ---')
print(classification_report(y_test, y_pred, target_names=LABELS, digits=4))

In [ ]:
# Export to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('mina_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite model saved: {len(tflite_model) / 1024:.1f} KB')

with open('labels.txt', 'w') as f:
    for label in LABELS:
        f.write(label + '\n')
print('Labels saved.')

# Verify TFLite model
interpreter = tf.lite.Interpreter(model_path='mina_classifier.tflite')
interpreter.allocate_tensors()
inp = interpreter.get_input_details()
out = interpreter.get_output_details()
print(f'Input: {inp[0]["shape"]}')
print(f'Output: {out[0]["shape"]}')

# Inference benchmark
import time
dummy = np.random.randn(1, 124, 40, 1).astype(np.float32)
times = []
for _ in range(1000):
    t0 = time.perf_counter()
    interpreter.set_tensor(inp[0]['index'], dummy)
    interpreter.invoke()
    times.append((time.perf_counter() - t0) * 1000)
print(f'\nInference benchmark (1000 runs):')
print(f'  Mean: {np.mean(times):.2f} ms')
print(f'  P95:  {np.percentile(times, 95):.2f} ms')

In [ ]:
# Save to Google Drive (so you don't lose it if runtime disconnects)
import shutil
drive_out = '/content/drive/MyDrive/minazap_models'
os.makedirs(drive_out, exist_ok=True)
shutil.copy('mina_classifier.tflite', drive_out)
shutil.copy('labels.txt', drive_out)
print(f'Model saved to Drive: {drive_out}/')

In [ ]:
# Download model files to your machine
from google.colab import files
files.download('mina_classifier.tflite')
files.download('labels.txt')